# RoadRisk AI - Sri Lanka Police Accident Data Analysis

**Purpose:** Generate proposal-ready figures from official Sri Lankan road-accident sources.

This notebook is designed for Google Colab. It supports manual upload of CSV/XLS/XLSX files, so it does not depend on fragile direct-download URLs.

### Recommended official sources
1. **2025 Traffic Reports (Sri Lanka Police):** https://www.police.lk/?p=21248  
2. **Road Accident Data by Severity of Injuries - 2012:** https://data.gov.lk/dataset/road-accident-data-severity-injuries-2012  
3. **Road Accident Data by Vehicle Type - 2012:** https://data.gov.lk/dataset/road-accident-data-vehicle-type-2012

For the two 2012 datasets, open the dataset page and choose the **CSV** resource. The Open Data Portal can be slow, so download the file in your browser first and then upload it here.

> **Important:** The 2012 datasets are historical context only. Do not present them as current accident conditions. Use the official 2025 Police report for the current national problem scale.


## 1. Setup
Run this cell first. All plots are saved into `/content/roadrisk_figures/` so you can insert them into the report later.

In [ ]:
# Core data-analysis libraries available in Google Colab.
import os
import re
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = '/content/roadrisk_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Use readable defaults without forcing a custom colour scheme.
plt.rcParams['figure.figsize'] = (9, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.2

print('Output folder:', OUTPUT_DIR)


## 2. Figure 1 - Official 2025 accident severity

These values come from the Sri Lanka Police **Summary Report on Road Traffic Accidents - 2025**:
- Fatal accidents: 2,583
- Serious accidents: 5,276
- Minor accidents: 10,067
- Damage-only accidents: 4,227
- Total accidents: 22,153
- Total deaths: 2,732

This chart does **not** require a dataset upload.

In [ ]:
# Official national totals from the Sri Lanka Police 2025 traffic report.
severity_2025 = pd.Series({
    'Fatal': 2583,
    'Serious': 5276,
    'Minor': 10067,
    'Damage-only': 4227,
}, name='Accidents')

total_accidents = int(severity_2025.sum())
total_deaths = 2732

ax = severity_2025.plot(kind='bar')
ax.set_title('Sri Lanka Road Traffic Accidents by Severity - 2025')
ax.set_xlabel('Accident severity')
ax.set_ylabel('Number of accidents')
ax.tick_params(axis='x', rotation=0)

# Add value labels above the bars.
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9,
                xytext=(0, 4), textcoords='offset points')

plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'fig_01_2025_accident_severity.png')
plt.savefig(path, dpi=220, bbox_inches='tight')
plt.show()

print(f'Total accidents: {total_accidents:,}')
print(f'Total deaths: {total_deaths:,}')
print('Saved:', path)


### Suggested interpretation for the report
Use factual wording such as:

> Sri Lanka Police recorded 22,153 road traffic accidents in 2025, including 2,583 fatal accidents and 2,732 deaths. Minor accidents formed the largest reported severity category. These figures demonstrate the scale of the road-safety problem, but they remain **post-event statistics**: they tell us what happened after a crash, not the near-miss signals that may have preceded it.

Do not infer causation from this chart.

## 3. Upload and analyse the 2012 injury-severity dataset

Download the CSV from:  
https://data.gov.lk/dataset/road-accident-data-severity-injuries-2012

Then run the upload cell below. The code also accepts XLS/XLSX.

In [ ]:
from google.colab import files

print('Upload the 2012 severity CSV/XLS/XLSX file now...')
uploaded = files.upload()
severity_filename = next(iter(uploaded))
print('Uploaded:', severity_filename)


In [ ]:
# Helper that loads CSV, XLS, or XLSX using the uploaded filename.
def load_table(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == '.csv':
        # sep=None lets pandas infer comma/tab/semicolon when possible.
        return pd.read_csv(path, sep=None, engine='python')
    if ext in ('.xls', '.xlsx'):
        return pd.read_excel(path)
    raise ValueError(f'Unsupported file type: {ext}')

severity_df = load_table(severity_filename)
severity_df.columns = [str(c).strip() for c in severity_df.columns]

print('Shape:', severity_df.shape)
display(severity_df.head())
print('\nColumns:')
for c in severity_df.columns:
    print('-', c)


In [ ]:
# Clean numeric columns while preserving the location/division name.
sev = severity_df.copy()

# Find a location-like column; otherwise use the first column.
location_candidates = [c for c in sev.columns if 'location' in c.lower() or 'district' in c.lower() or 'division' in c.lower()]
location_col = location_candidates[0] if location_candidates else sev.columns[0]

# Convert every non-location column to numeric where possible.
for c in sev.columns:
    if c != location_col:
        sev[c] = pd.to_numeric(sev[c].astype(str).str.replace(',', '', regex=False), errors='coerce')

# Remove the national Total row for district/division comparisons.
sev_no_total = sev[~sev[location_col].astype(str).str.strip().str.lower().eq('total')].copy()

# Identify male/female columns by keywords. These names match the official 2012 CSV,
# but the code is intentionally tolerant of small naming differences.
def find_cols(*terms):
    out=[]
    for c in sev_no_total.columns:
        lc=c.lower()
        if all(t in lc for t in terms):
            out.append(c)
    return out

death_cols = find_cols('death')
grievous_cols = find_cols('grievous')
# "Non Grievous" also contains the word grievous, so isolate it explicitly.
non_grievous_cols = [c for c in sev_no_total.columns if 'non' in c.lower() and 'grievous' in c.lower()]
grievous_only_cols = [c for c in grievous_cols if c not in non_grievous_cols]

sev_no_total['Deaths'] = sev_no_total[death_cols].sum(axis=1, min_count=1) if death_cols else np.nan
sev_no_total['Grievous injuries'] = sev_no_total[grievous_only_cols].sum(axis=1, min_count=1) if grievous_only_cols else np.nan
sev_no_total['Non-grievous injuries'] = sev_no_total[non_grievous_cols].sum(axis=1, min_count=1) if non_grievous_cols else np.nan

print('Location column:', location_col)
print('Death columns:', death_cols)
print('Grievous columns:', grievous_only_cols)
print('Non-grievous columns:', non_grievous_cols)
display(sev_no_total[[location_col,'Deaths','Grievous injuries','Non-grievous injuries']].head())


### Figure 2 - Highest recorded casualty burden by police division (2012 historical context)
This is useful for showing that official Sri Lankan open data can be analysed geographically, but clearly label the year as **2012**.

In [ ]:
plot_cols = ['Deaths','Grievous injuries','Non-grievous injuries']
available_plot_cols = [c for c in plot_cols if c in sev_no_total and sev_no_total[c].notna().any()]

if not available_plot_cols:
    raise ValueError('Could not identify the expected injury columns. Review the printed column names above.')

sev_no_total['Calculated casualties'] = sev_no_total[available_plot_cols].sum(axis=1)
top = sev_no_total.nlargest(10, 'Calculated casualties').set_index(location_col)

ax = top[available_plot_cols].plot(kind='barh', stacked=True, figsize=(10, 7))
ax.set_title('Top 10 Police Divisions by Recorded Casualties - 2012')
ax.set_xlabel('Recorded deaths and injuries')
ax.set_ylabel('Police division / location')
ax.invert_yaxis()
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'fig_02_2012_casualties_top10.png')
plt.savefig(path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', path)


## 4. Upload and analyse the 2012 vehicle-type dataset

Download the CSV from:  
https://data.gov.lk/dataset/road-accident-data-vehicle-type-2012

Then upload it below.

In [ ]:
print('Upload the 2012 vehicle-type CSV/XLS/XLSX file now...')
uploaded_vehicle = files.upload()
vehicle_filename = next(iter(uploaded_vehicle))
print('Uploaded:', vehicle_filename)


In [ ]:
vehicle_df = load_table(vehicle_filename)
vehicle_df.columns = [str(c).strip() for c in vehicle_df.columns]
print('Shape:', vehicle_df.shape)
display(vehicle_df.head())
print('\nColumns:')
for c in vehicle_df.columns:
    print('-', c)


### Figure 3 - Vehicle categories represented in 2012 accident records
The official dataset contains a national **Total** row. This code uses that row where available, avoiding double-counting.

In [ ]:
veh = vehicle_df.copy()
location_candidates = [c for c in veh.columns if 'location' in c.lower() or 'district' in c.lower() or 'division' in c.lower()]
veh_location = location_candidates[0] if location_candidates else veh.columns[0]

for c in veh.columns:
    if c != veh_location:
        veh[c] = pd.to_numeric(veh[c].astype(str).str.replace(',', '', regex=False), errors='coerce')

# Prefer the official Total row. If it is missing, sum rows as a fallback.
total_mask = veh[veh_location].astype(str).str.strip().str.lower().eq('total')
if total_mask.any():
    national = veh.loc[total_mask].iloc[0]
else:
    national = veh.drop(columns=[veh_location]).sum(numeric_only=True)

ignore_names = {'total'}
vehicle_totals = {}
for c in veh.columns:
    if c == veh_location or c.strip().lower() in ignore_names:
        continue
    value = pd.to_numeric(pd.Series([national[c]]), errors='coerce').iloc[0]
    if pd.notna(value):
        vehicle_totals[c] = float(value)

vehicle_series = pd.Series(vehicle_totals).sort_values(ascending=False)

# Keep the top 10 vehicle categories for readability.
top_vehicles = vehicle_series.head(10).sort_values()
ax = top_vehicles.plot(kind='barh', figsize=(10, 6.5))
ax.set_title('Vehicle Categories in Sri Lanka Police Road-Accident Data - 2012')
ax.set_xlabel('Recorded vehicle involvements')
ax.set_ylabel('Vehicle type')
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'fig_03_2012_vehicle_types.png')
plt.savefig(path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', path)


## 5. Export all figures
Run this at the end to download a ZIP containing every graph generated by this notebook.

In [ ]:
zip_path = '/content/roadrisk_police_figures.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        z.write(os.path.join(OUTPUT_DIR, fname), arcname=fname)

print('Created:', zip_path)
files.download(zip_path)


## What to add to the report after running this notebook

1. Insert **Figure 1** under the current-problem section and discuss the 2025 national scale.
2. Insert **Figure 2** only as historical context; state clearly that the open dataset is from 2012.
3. Insert **Figure 3** if it helps motivate mixed vehicle classes such as motorcycles and three-wheelers.
4. Do not claim that 2012 patterns are unchanged in 2026.
5. Replace report placeholders with your actual figure files and add 2-4 sentences of interpretation beneath each figure.
